In [ ]:
!pip install -U sentence-transformers


In [ ]:

!pip -q install -U "sentence-transformers==2.5.1" "transformers==4.43.3" "huggingface_hub==0.24.6" faiss-cpu
import torch, random, numpy as np, os, pathlib
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, pickle, random, torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, InputExample

DATASET_NAME = "data-train"               # tên dataset trong /kaggle/input/
PKL_FILE     = "data_train_vnexpress.pkl" # file dữ liệu .pkl

INPUT_FILE_PATH = Path(f"/kaggle/input/{DATASET_NAME}/{PKL_FILE}")
OUTPUT_DIR = Path("/kaggle/working/bkai_finetuned_vn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Input:", INPUT_FILE_PATH.exists(), "|", INPUT_FILE_PATH)

# Load triplets và thêm prefix chuẩn E5
def load_triplets(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    triplets = []
    for ex in data:
        if isinstance(ex, InputExample) and len(ex.texts) >= 3:
            q = f"query: {ex.texts[0].strip()}"
            p = f"passage: {ex.texts[1].strip()}"
            n = f"passage: {ex.texts[2].strip()}"
            triplets.append(InputExample(texts=[q, p, n]))
    print(f"✅ Tổng số mẫu hợp lệ: {len(triplets)}")
    return triplets

triplets = load_triplets(INPUT_FILE_PATH)

# Chia train / val / test
train_triplets, temp_triplets = train_test_split(triplets, test_size=0.2, random_state=42)
val_triplets, test_triplets = train_test_split(temp_triplets, test_size=0.5, random_state=42)

print(f"Train: {len(train_triplets)} | Val: {len(val_triplets)} | Test: {len(test_triplets)}")


In [ ]:
import torch
import torch.nn.functional as F

loss_history = []

class LoggingTripletLoss(torch.nn.Module):
    def __init__(self, model, margin=0.2, history_list=None, log_every=5):
        super().__init__()
        self.model = model
        self.margin = margin
        self.history = history_list if history_list is not None else []
        self.log_every = log_every
        self._step = 0

    def forward(self, sentence_features, labels=None):
        emb_anchor = self.model(sentence_features[0])['sentence_embedding']
        emb_pos    = self.model(sentence_features[1])['sentence_embedding']
        emb_neg    = self.model(sentence_features[2])['sentence_embedding']

        pos_dist = 1 - F.cosine_similarity(emb_anchor, emb_pos)
        neg_dist = 1 - F.cosine_similarity(emb_anchor, emb_neg)
        loss = F.relu(pos_dist - neg_dist + self.margin).mean()

        self._step += 1
        if self._step % self.log_every == 0:
            self.history.append((self._step, float(loss.detach().cpu().item())))

        return loss


In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers.evaluation import SentenceEvaluator
from sentence_transformers.util import cos_sim
import math

# Model base BKAI (có thể đổi thành model HuggingFace khác)
BASE_MODEL = "bkai-foundation-models/vietnamese-bi-encoder"
BATCH_SIZE = 4
EPOCHS = 10
LR = 2e-5
MAX_LEN = 256
WARMUP_STEPS = 100
EVAL_STEPS = 200
USE_FP16 = True

# Load model BKAI
model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_LEN

# DataLoader
train_loader = DataLoader(train_triplets, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# Loss
train_loss = LoggingTripletLoss(model, margin=0.2, history_list=loss_history, log_every=5)

# Evaluator nhỏ
class TinyEvaluator(SentenceEvaluator):
    def __init__(self, samples, max_samples=512):
        self.samples = samples[:max_samples]
    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        q = [s.texts[0] for s in self.samples]
        p = [s.texts[1] for s in self.samples]
        n = [s.texts[2] for s in self.samples]
        with torch.no_grad():
            q_emb = model.encode(q, normalize_embeddings=True, convert_to_tensor=True)
            p_emb = model.encode(p, normalize_embeddings=True, convert_to_tensor=True)
            n_emb = model.encode(n, normalize_embeddings=True, convert_to_tensor=True)
            acc = (cos_sim(q_emb, p_emb).diagonal() > cos_sim(q_emb, n_emb).diagonal()).float().mean().item()
        print(f"[Eval] epoch={epoch} step={steps} acc_pos>neg={acc:.4f}")
        return acc

evaluator = TinyEvaluator(val_triplets, max_samples=512)

print("🚀 Bắt đầu huấn luyện BKAI...")
model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=EPOCHS,
    optimizer_params={'lr': LR},
    warmup_steps=WARMUP_STEPS,
    output_path=str(OUTPUT_DIR),
    evaluation_steps=EVAL_STEPS,
    evaluator=evaluator,
    save_best_model=True,
    show_progress_bar=True,
    use_amp=USE_FP16
)
print("✅ Huấn luyện hoàn tất. Model lưu tại:", OUTPUT_DIR)


In [ ]:
import matplotlib.pyplot as plt

steps = [s for s, _ in loss_history]
loss_vals = [v for _, v in loss_history]

plt.figure(figsize=(8,5))
plt.plot(steps, loss_vals, color="red", label="Train Loss")
plt.title("Training Loss Curve (BKAI Fine-tune)")
plt.xlabel("Training Steps")
plt.ylabel("Loss Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.savefig("/kaggle/working/bkai_loss_curve.png", dpi=300)
print("📈 Đã lưu biểu đồ tại: /kaggle/working/bkai_loss_curve.png")


In [ ]:
from sentence_transformers.util import cos_sim
import math, pandas as pd, random

def dcg_at_k(rank, k): return 1.0 / math.log2(rank + 1) if rank <= k else 0.0

def evaluate_model(model, test_triplets, cand_topk=32, max_queries=200):
    all_negs = [ex.texts[2] for ex in test_triplets]
    results = {"MRR":0,"Acc@1":0,"Acc@3":0,"Acc@5":0,"DCG@1":0,"DCG@3":0,"DCG@5":0,"AvgRank":0}
    for ex in random.sample(test_triplets, min(max_queries, len(test_triplets))):
        q, pos = ex.texts[0], ex.texts[1]
        negs = random.sample(all_negs, min(cand_topk-1, len(all_negs)))
        cands = [pos] + negs
        qe = model.encode(q, normalize_embeddings=True, convert_to_tensor=True)
        pes = model.encode(cands, normalize_embeddings=True, convert_to_tensor=True)
        sims = cos_sim(qe, pes).flatten()
        rank = (torch.argsort(sims, descending=True) == 0).nonzero(as_tuple=False).item() + 1
        results["MRR"] += 1/rank
        results["AvgRank"] += rank
        for k in [1,3,5]:
            results[f"Acc@{k}"] += rank <= k
            results[f"DCG@{k}"] += dcg_at_k(rank,k)
    n = min(max_queries, len(test_triplets))
    for k in results: results[k] /= n
    return results

# So sánh BKAI gốc vs fine-tuned
model_base = SentenceTransformer(BASE_MODEL)
res_base = evaluate_model(model_base, test_triplets)
res_ft   = evaluate_model(model, test_triplets)

print("📊 Hiệu quả mô hình BKAI gốc:")
print(res_base)
print("\n📊 Hiệu quả mô hình BKAI sau fine-tune:")
print(res_ft)

pd.DataFrame([
    {"Metric": k, "BKAI Base": res_base[k], "Fine-tuned": res_ft[k], "Δ (FT-Base)": res_ft[k]-res_base[k]}
    for k in ["MRR","Acc@1","Acc@3","Acc@5","DCG@1","DCG@3","DCG@5","AvgRank"]
])
